# 01 · Ingestion Pipeline: Load → Split → Embed → Store
**Concept:** the offline half of RAG. Turn four document formats into chunks with metadata, embed them, and persist them in Chroma.
**TAC scenario:** indexing KB articles (.md), release notes (.md), resolved tickets (.json) and product guides (.pdf).

> **Prereqs (once):** `uv sync` → copy `.env.example` to `.env` and add your key → `uv run python scripts/generate_corpus.py` → select the `.venv` kernel.

In [ ]:
import os, json, re
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent
DATA = ROOT / "data"
load_dotenv(ROOT / ".env")
EMBED_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

assert (DATA / "manifest.json").exists(), "Run: uv run python scripts/generate_corpus.py"
print(json.loads((DATA / "manifest.json").read_text())["counts"])

## 1. Load: Markdown with front-matter metadata

In [ ]:
from langchain_core.documents import Document

def parse_front_matter(text: str) -> tuple[dict, str]:
    m = re.match(r"---\n(.*?)\n---\n", text, re.S)
    meta = {}
    for line in m.group(1).splitlines():
        k, v = line.split(":", 1)
        v = v.strip()
        meta[k.strip()] = json.loads(v) if v.startswith('"') else v
    return meta, text[m.end():]

def load_markdown_dir(subdir: str) -> list[Document]:
    docs = []
    for path in sorted((DATA / subdir).glob("*.md")):
        meta, body = parse_front_matter(path.read_text(encoding="utf-8"))
        docs.append(Document(page_content=body.strip(), metadata={**meta, "source": f"{subdir}/{path.name}"}))
    return docs

kb_docs = load_markdown_dir("kb")
rn_docs = load_markdown_dir("release_notes")
print(len(kb_docs), "KB articles |", len(rn_docs), "release notes")
print(kb_docs[0].metadata)
print(kb_docs[0].page_content[:300])

## 2. Load: JSON tickets (you decide what is *content* vs *metadata*)

In [ ]:
tickets = json.loads((DATA / "tickets" / "resolved_tickets.json").read_text(encoding="utf-8"))
family = lambda v: ".".join(v.split(".")[:2])

ticket_docs = [Document(
    page_content=f"{t['subject']}\n{t['description']}\nResolution: {t['resolution']}",   # what gets embedded
    metadata={"doc_id": t["ticket_id"], "doc_type": "ticket", "product": t["product"], "version": t["version"],
              "version_family": family(t["version"]), "kb_refs": ",".join(t["kb_refs"]),   # Chroma metadata must be scalar
              "error_code": t["error_code"] or "none", "source": "tickets/resolved_tickets.json"},
) for t in tickets]
print(len(ticket_docs), "tickets")
ticket_docs[0]

## 3. Load: PDF product guides with `PyPDFLoader` (one Document per page)

> `langchain-community` may print a *deprecation* warning: the package is being wound down, but `PyPDFLoader` and `BM25Retriever` still live there and work. Safe to ignore for this course.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PRODUCT = {"AR": "Aurora AR-900", "NF": "Nimbus NF-300", "OW": "Orion OW-20", "AT": "Atlas SD-WAN", "HX": "Helix HX-5000"}
guide_docs = []
for path in sorted((DATA / "guides").glob("*.pdf")):
    for page in PyPDFLoader(str(path)).load():
        page.metadata = {"doc_id": path.stem, "doc_type": "guide", "product": PRODUCT[path.stem.split("-")[1]],
                         "version_family": "all", "page": page.metadata.get("page", 0) + 1, "source": f"guides/{path.name}"}
        guide_docs.append(page)
print(len(guide_docs), "guide pages")
print(guide_docs[1].metadata, "\n", guide_docs[1].page_content[:300])

In [ ]:
from collections import Counter
all_docs = kb_docs + rn_docs + ticket_docs + guide_docs
Counter(d.metadata["doc_type"] for d in all_docs)

## 4. Split into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(all_docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = f"{c.metadata['doc_id']}#{i}"

sizes = [len(c.page_content) for c in chunks]
print(f"{len(all_docs)} documents → {len(chunks)} chunks | avg {sum(sizes)//len(sizes)} chars | max {max(sizes)}")
print(chunks[0].metadata)    # metadata is copied onto every chunk

## 5. Embed: text → vector

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
v = embeddings.embed_query("BGP sessions flap after upgrade")
print(len(v), "dimensions:", [round(x, 4) for x in v[:5]], "…")

In [ ]:
import math
def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

phrases = ["link flap on the uplink", "interface keeps bouncing", "license renewal quote"]
vecs = embeddings.embed_documents(phrases)
print(f"'{phrases[0]}' vs '{phrases[1]}': {cosine(vecs[0], vecs[1]):.3f}   ← similar meaning, different words")
print(f"'{phrases[0]}' vs '{phrases[2]}': {cosine(vecs[0], vecs[2]):.3f}   ← unrelated")

## 6. Store: persist the chunks in Chroma

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = str(ROOT / ".chroma")
Chroma(collection_name="tac_corpus", embedding_function=embeddings, persist_directory=CHROMA_DIR).delete_collection()  # fresh start

vectorstore = Chroma.from_documents(
    chunks, embeddings,
    ids=[c.metadata["chunk_id"] for c in chunks],
    collection_name="tac_corpus",
    persist_directory=CHROMA_DIR,
)
print("Stored chunks:", vectorstore._collection.count(), "→", CHROMA_DIR)

In [ ]:
# Quick smoke test of the stored index
for d in vectorstore.similarity_search("tunnels drop at the same time every night", k=3):
    print(d.metadata["doc_id"], "|", d.metadata["doc_type"], "|", " ".join(d.page_content.split())[:80])

> Notebooks 02-11 reopen this store through `rag_common.get_vectorstore()`, which contains the same code as this notebook.

---
**🔧 TAC takeaway:** Ingestion is ticket intake for knowledge: tag product, version and source once, and every later step (filters, citations, routing) depends on getting it right.